In [1]:
import sklearn
import numpy as np
import pandas as pd
import joblib

print("sklearn:", sklearn.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)


sklearn: 1.8.0
numpy: 2.4.0
pandas: 2.3.3


In [2]:
# ============================================================================
# BEARING FAILURE PREDICTION - LATEST VERSIONS COMPATIBLE
# ============================================================================

import numpy as np
import pandas as pd
import joblib
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from xgboost import XGBClassifier

print("="*60)
print("MODEL TRAINING WITH LATEST VERSIONS")
print("="*60)
print(f"scikit-learn: 1.8.0")
print(f"numpy: 2.4.0")
print(f"pandas: 2.3.3")
print("="*60)

MODEL TRAINING WITH LATEST VERSIONS
scikit-learn: 1.8.0
numpy: 2.4.0
pandas: 2.3.3


In [3]:
def load_and_prepare_data(csv_path):
    """
    Load CSV data and prepare for training
    """
    print("\n" + "="*60)
    print("LOADING DATA")
    print("="*60)
    
    try:
        # Load CSV with new pandas version
        df = pd.read_csv(csv_path)
        print(f"✓ Data loaded: {df.shape}")
        
        # Check required columns
        required_cols = ['Label', 'RMS', 'Kurtosis', 'Peak']
        missing_cols = [col for col in required_cols if col not in df.columns]
        
        if missing_cols:
            print(f"✗ Missing columns: {missing_cols}")
            return None, None, None
        
        # Feature columns (update based on your CSV)
        feature_columns = [
            'RMS', 'Kurtosis', 'Skewness', 'Peak', 'Variance',
            'Mean_Abs', 'Peak_to_Peak', 'Crest_Factor', 'Shape_Factor',
            'Impulse_Factor', 'Spectral_Centroid', 'Spectral_Spread',
            'Dominant_Freq', 'Dominant_Mag', 'RMS_MA10', 'RMS_Trend',
            'Kurtosis_MA10', 'Kurtosis_Trend', 'Peak_MA10', 'Peak_Trend'
        ]
        
        # Filter only existing columns
        feature_columns = [col for col in feature_columns if col in df.columns]
        print(f"✓ Using {len(feature_columns)} features")
        
        # Split features and target
        X = df[feature_columns].values
        y = df['Label'].values
        
        print(f"✓ Features shape: {X.shape}")
        print(f"✓ Target shape: {y.shape}")
        print(f"✓ Class distribution: {np.bincount(y)}")
        
        return X, y, feature_columns, df
        
    except Exception as e:
        print(f"✗ Error loading data: {e}")
        return None, None, None, None

# Load your data
CSV_PATH = "bearing_features.csv"  # Update this path
X, y, feature_columns, df = load_and_prepare_data(CSV_PATH)


LOADING DATA
✓ Data loaded: (46480, 27)
✓ Using 20 features
✓ Features shape: (46480, 20)
✓ Target shape: (46480,)
✓ Class distribution: [44400  2080]


In [4]:
def train_models_with_new_versions(X, y, feature_columns):
    """
    Train models using latest sklearn 1.8.0 features
    """
    print("\n" + "="*60)
    print("TRAINING MODELS (sklearn 1.8.0)")
    print("="*60)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"Training samples: {X_train.shape[0]}")
    print(f"Test samples: {X_test.shape[0]}")
    
    # Define models with latest parameters
    models = {
        'RandomForest': RandomForestClassifier(
            n_estimators=200,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
            class_weight='balanced'
        ),
        
        'XGBoost': XGBClassifier(
            n_estimators=200,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric='logloss',
            use_label_encoder=False,
            scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1])
        ),
        
        'GradientBoosting': GradientBoostingClassifier(
            n_estimators=150,
            learning_rate=0.1,
            max_depth=7,
            random_state=42,
            subsample=0.8
        ),
        
        'SVM': SVC(
            C=1.0,
            kernel='rbf',
            probability=True,
            random_state=42,
            class_weight='balanced'
        ),
        
        'NeuralNetwork': MLPClassifier(
            hidden_layer_sizes=(128, 64, 32),
            activation='relu',
            solver='adam',
            alpha=0.0001,
            batch_size=32,
            learning_rate='adaptive',
            max_iter=500,
            random_state=42,
            early_stopping=True,
            n_iter_no_change=10
        )
    }
    
    # Train and evaluate
    results = {}
    
    for name, model in models.items():
        print(f"\n--- Training {name} ---")
        
        # Train
        model.fit(X_train_scaled, y_train)
        
        # Predict
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        
        # Metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        roc_auc = roc_auc_score(y_test, y_proba)
        
        # Store results
        results[name] = {
            'model': model,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'roc_auc': roc_auc,
            'y_pred': y_pred,
            'y_proba': y_proba
        }
        
        print(f"  Accuracy:  {accuracy:.4f}")
        print(f"  F1-Score:  {f1:.4f}")
        print(f"  ROC-AUC:   {roc_auc:.4f}")
    
    return results, X_test_scaled, y_test, scaler

# Train models
results, X_test_scaled, y_test, scaler = train_models_with_new_versions(
    X, y, feature_columns
)


TRAINING MODELS (sklearn 1.8.0)
Training samples: 37184
Test samples: 9296

--- Training RandomForest ---
  Accuracy:  0.9777
  F1-Score:  0.7953
  ROC-AUC:   0.9934

--- Training XGBoost ---
  Accuracy:  0.9812
  F1-Score:  0.8187
  ROC-AUC:   0.9962

--- Training GradientBoosting ---
  Accuracy:  0.9854
  F1-Score:  0.8313
  ROC-AUC:   0.9961

--- Training SVM ---
  Accuracy:  0.9064
  F1-Score:  0.4828
  ROC-AUC:   0.9857

--- Training NeuralNetwork ---
  Accuracy:  0.9802
  F1-Score:  0.7401
  ROC-AUC:   0.9933


In [10]:
# ============================================================================
# UPDATED: SAVE FEATURES IN JSON FORMAT
# ============================================================================

def save_model_components_with_json(results, scaler, feature_columns, df_metadata=None):
    """
    Save model components with features in JSON format
    """
    print("\n" + "="*60)
    print("SAVING MODEL COMPONENTS (JSON FORMAT)")
    print("="*60)
    
    # Select best model
    best_model_name = max(results.items(), key=lambda x: x[1]['f1'])[0]
    best_model = results[best_model_name]['model']
    
    print(f"✓ Best model: {best_model_name}")
    print(f"  F1-Score: {results[best_model_name]['f1']:.4f}")
    print(f"  Accuracy: {results[best_model_name]['accuracy']:.4f}")
    
    # Create timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    version = "v1"
    
    # ============================================
    # 1. SAVE SCALER (Joblib format - Binary)
    # ============================================
    scaler_filename = f"scaler_{version}.joblib"
    joblib.dump(scaler, scaler_filename)
    print(f"✓ Scaler saved: {scaler_filename}")
    
    # ============================================
    # 2. SAVE MODEL (Joblib format - Binary)
    # ============================================
    model_filename = f"model_{version}.joblib"
    joblib.dump(best_model, model_filename)
    print(f"✓ Model saved: {model_filename}")
    
    # ============================================
    # 3. SAVE FEATURES IN JSON FORMAT
    # ============================================
    # Create comprehensive feature information
    features_info = {
        'feature_columns': feature_columns,
        'feature_count': len(feature_columns),
        'feature_categories': categorize_features(feature_columns),
        'feature_descriptions': get_feature_descriptions(feature_columns),
        'default_values': get_default_values(feature_columns),
        'data_types': infer_data_types(df_metadata, feature_columns) if df_metadata is not None else {},
        'created_at': datetime.now().isoformat(),
        'version': version,
        'model_compatible': True
    }
    
    features_json_filename = f"features_{version}.json"
    with open(features_json_filename, 'w', encoding='utf-8') as f:
        json.dump(features_info, f, indent=4, ensure_ascii=False)
    
    print(f"✓ Features saved (JSON): {features_json_filename}")
    
    # Also save as simple list for easy access
    simple_features = {
        'features': feature_columns,
        'count': len(feature_columns)
    }
    
    simple_features_filename = f"features_list_{version}.json"
    with open(simple_features_filename, 'w', encoding='utf-8') as f:
        json.dump(simple_features, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Simple features list saved: {simple_features_filename}")
    
    # ============================================
    # 4. SAVE COMPLETE PACKAGE (Backup)
    # ============================================
    model_package = {
        'model': best_model,
        'scaler': scaler,
        'feature_columns': feature_columns,
        'model_name': best_model_name,
        'model_type': type(best_model).__name__,
        'versions': {
            'sklearn': '1.8.0',
            'numpy': '2.4.0',
            'pandas': '2.3.3',
            'python': sys.version.split()[0]
        },
        'training_date': datetime.now().isoformat(),
        'performance': {
            'accuracy': float(results[best_model_name]['accuracy']),
            'precision': float(results[best_model_name]['precision']),
            'recall': float(results[best_model_name]['recall']),
            'f1_score': float(results[best_model_name]['f1']),
            'roc_auc': float(results[best_model_name]['roc_auc'])
        },
        'training_info': {
            'train_samples': X_train.shape[0],
            'test_samples': X_test.shape[0],
            'feature_count': len(feature_columns)
        }
    }
    
    package_filename = f"model_package_{version}.joblib"
    joblib.dump(model_package, package_filename, compress=3)
    print(f"✓ Complete package saved: {package_filename}")
    
    # ============================================
    # 5. SAVE DETAILED METADATA AS JSON
    # ============================================
    metadata = {
        'model': {
            'name': best_model_name,
            'type': type(best_model).__name__,
            'algorithm': best_model_name,
            'parameters': get_model_params(best_model),
            'training_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'version': version
        },
        'files': {
            'model': model_filename,
            'scaler': scaler_filename,
            'features_json': features_json_filename,
            'features_simple': simple_features_filename,
            'package': package_filename
        },
        'performance': model_package['performance'],
        'requirements': model_package['versions'],
        'features': {
            'count': len(feature_columns),
            'list': feature_columns,
            'categories': features_info['feature_categories']
        },
        'dataset': {
            'total_samples': len(df_metadata) if df_metadata is not None else 'unknown',
            'training_samples': X_train.shape[0],
            'test_samples': X_test.shape[0],
            'class_distribution': get_class_distribution(y) if 'y' in locals() else {}
        },
        'deployment': {
            'django_compatible': True,
            'recommended_structure': 'ml_models/',
            'loading_method': 'use ml_predictor.py'
        }
    }
    
    metadata_filename = f"model_metadata_{version}.json"
    with open(metadata_filename, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False, default=str)
    
    print(f"✓ Metadata saved: {metadata_filename}")
    
    # ============================================
    # 6. CREATE REQUIREMENTS.TXT
    # ============================================
    create_django_requirements_json()
    
    # ============================================
    # 7. CREATE UPDATED DJANGO LOADER SCRIPT
    # ============================================
    create_django_loader_script_json(feature_columns, version)
    
    # ============================================
    # 8. CREATE CONFIG FILE FOR DJANGO
    # ============================================
    create_django_config_file(feature_columns, version)
    
    print("\n" + "="*60)
    print("ALL FILES SAVED SUCCESSFULLY!")
    print("="*60)
    
    # Display file summary
    print(f"""
Generated Files:
═══════════════════════════════════════════════════════════════════════
1. {model_filename:40} → Trained ML Model (binary)
2. {scaler_filename:40} → Feature Scaler (binary)
3. {features_json_filename:40} → Features with metadata (JSON)
4. {simple_features_filename:40} → Simple features list (JSON)
5. {package_filename:40} → Complete package (backup)
6. {metadata_filename:40} → Detailed metadata (JSON)
7. requirements_django.txt           → Django requirements
8. ml_predictor.py                   → Django loader script
9. ml_config.json                    → Django configuration
═══════════════════════════════════════════════════════════════════════
""")
    
    return {
        'model_file': model_filename,
        'scaler_file': scaler_filename,
        'features_json_file': features_json_filename,
        'features_simple_file': simple_features_filename,
        'metadata_file': metadata_filename
    }


# ============================================================================
# HELPER FUNCTIONS FOR JSON FEATURES
# ============================================================================

def categorize_features(feature_columns):
    """Categorize features into groups"""
    categories = {
        'time_domain': [],
        'frequency_domain': [],
        'statistical': [],
        'rolling': [],
        'other': []
    }
    
    time_keywords = ['RMS', 'Peak', 'Variance', 'Mean_Abs', 'Peak_to_Peak']
    freq_keywords = ['Spectral', 'Dominant', 'Freq', 'Mag']
    stat_keywords = ['Kurtosis', 'Skewness', 'Shape', 'Impulse', 'Crest']
    rolling_keywords = ['MA10', 'Trend', 'Rolling']
    
    for feature in feature_columns:
        feature_lower = feature.lower()
        
        if any(keyword.lower() in feature_lower for keyword in rolling_keywords):
            categories['rolling'].append(feature)
        elif any(keyword.lower() in feature_lower for keyword in time_keywords):
            categories['time_domain'].append(feature)
        elif any(keyword.lower() in feature_lower for keyword in freq_keywords):
            categories['frequency_domain'].append(feature)
        elif any(keyword.lower() in feature_lower for keyword in stat_keywords):
            categories['statistical'].append(feature)
        else:
            categories['other'].append(feature)
    
    return categories

def get_feature_descriptions(feature_columns):
    """Get descriptions for each feature"""
    descriptions = {
        'RMS': 'Root Mean Square - Overall vibration level',
        'Kurtosis': 'Peakedness of vibration signal',
        'Skewness': 'Asymmetry of vibration distribution',
        'Peak': 'Maximum absolute vibration amplitude',
        'Variance': 'Spread of vibration signal',
        'Mean_Abs': 'Mean absolute value of vibration',
        'Peak_to_Peak': 'Difference between max and min vibration',
        'Crest_Factor': 'Ratio of peak to RMS value',
        'Shape_Factor': 'Ratio of RMS to mean absolute value',
        'Impulse_Factor': 'Ratio of peak to mean absolute value',
        'Spectral_Centroid': 'Center of mass of frequency spectrum',
        'Spectral_Spread': 'Spread of frequency spectrum',
        'Dominant_Freq': 'Frequency with maximum amplitude',
        'Dominant_Mag': 'Amplitude at dominant frequency',
        'RMS_MA10': '10-point moving average of RMS',
        'RMS_Trend': 'Trend of RMS values over time',
        'Kurtosis_MA10': '10-point moving average of Kurtosis',
        'Kurtosis_Trend': 'Trend of Kurtosis values over time',
        'Peak_MA10': '10-point moving average of Peak values',
        'Peak_Trend': 'Trend of Peak values over time'
    }
    
    feature_descriptions = {}
    for feature in feature_columns:
        feature_descriptions[feature] = descriptions.get(feature, 'Vibration signal feature')
    
    return feature_descriptions

def get_default_values(feature_columns):
    """Get sensible default values for features"""
    defaults = {}
    
    for feature in feature_columns:
        if 'RMS' in feature or 'Mean' in feature:
            defaults[feature] = 0.0
        elif 'Kurtosis' in feature:
            defaults[feature] = 0.0  # Normal distribution has kurtosis ≈ 0
        elif 'Skewness' in feature:
            defaults[feature] = 0.0  # Symmetric distribution
        elif 'Peak' in feature or 'Amplitude' in feature:
            defaults[feature] = 0.0
        elif 'Freq' in feature:
            defaults[feature] = 0.0
        elif 'Trend' in feature:
            defaults[feature] = 0.0  # No trend
        else:
            defaults[feature] = 0.0
    
    return defaults

def infer_data_types(df, feature_columns):
    """Infer data types from dataframe"""
    if df is None:
        return {}
    
    data_types = {}
    for feature in feature_columns:
        if feature in df.columns:
            dtype = str(df[feature].dtype)
            data_types[feature] = dtype
    
    return data_types

def get_model_params(model):
    """Get model parameters"""
    try:
        return model.get_params()
    except:
        return str(model)

def get_class_distribution(y):
    """Get class distribution"""
    unique, counts = np.unique(y, return_counts=True)
    distribution = {}
    for label, count in zip(unique, counts):
        distribution[f'class_{int(label)}'] = {
            'count': int(count),
            'percentage': float(count/len(y)*100)
        }
    return distribution

def create_django_requirements_json():
    """Create requirements.txt for Django with JSON support"""
    requirements = """# Django Bearing Failure Detection - Requirements
# Generated on: {}
# For use with JSON feature files

# =========== CORE ML LIBRARIES ===========
scikit-learn==1.8.0
numpy==2.4.0
pandas==2.3.3
joblib==1.4.2
xgboost==2.1.2
scipy==1.14.1

# =========== DJANGO FRAMEWORK ===========
Django==5.0.4
djangorestframework==3.15.1

# =========== JSON & API SUPPORT ===========
django-cors-headers==4.3.1
drf-yasg==1.21.7

# =========== OPTIONAL UTILITIES ===========
python-dateutil==2.8.2
pytz==2023.3
""".format(datetime.now().strftime("%Y-%m-%d"))
    
    with open("requirements_django.txt", "w") as f:
        f.write(requirements)

def create_django_loader_script_json(feature_columns, version):
    """Create Django loader script that uses JSON features"""
    script = '''"""
File: ml_predictor.py
Django ML Model Loader for Bearing Failure Detection
Uses JSON format for features
"""

import joblib
import json
import numpy as np
import pandas as pd
import os
from pathlib import Path

class BearingFailurePredictor:
    """
    Django-compatible predictor with JSON feature configuration
    """
    
    def __init__(self, model_dir="ml_models", config_file="ml_config.json"):
        """
        Initialize predictor using JSON configuration
        
        Args:
            model_dir: Directory containing model files
            config_file: JSON configuration file name
        """
        self.model_dir = Path(model_dir)
        self.config_file = config_file
        
        # Load configuration
        self.config = self._load_config()
        
        # Load components using config
        self.model = self._load_component(self.config['files']['model'])
        self.scaler = self._load_component(self.config['files']['scaler'])
        
        # Load features from JSON
        features_path = self.model_dir / self.config['files']['features_json']
        with open(features_path, 'r', encoding='utf-8') as f:
            self.features_data = json.load(f)
        
        self.feature_columns = self.features_data['feature_columns']
        self.feature_descriptions = self.features_data.get('feature_descriptions', {})
        self.default_values = self.features_data.get('default_values', {})
        
        print(f"✓ Model: {self.config['model']['name']}")
        print(f"✓ Features: {len(self.feature_columns)}")
        print(f"✓ Scaler: {type(self.scaler).__name__}")
        print(f"✓ Version: {self.config['model']['version']}")
        
        self.is_loaded = True
    
    def _load_config(self):
        """Load configuration from JSON file"""
        config_path = self.model_dir / self.config_file
        if not config_path.exists():
            # Fallback to default config
            return self._create_default_config()
        
        with open(config_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    
    def _create_default_config(self):
        """Create default configuration"""
        return {
            'model': {
                'name': 'Bearing Failure Detector',
                'version': 'v1',
                'type': 'RandomForest'
            },
            'files': {
                'model': 'model_v1.joblib',
                'scaler': 'scaler_v1.joblib',
                'features_json': 'features_v1.json',
                'metadata': 'model_metadata_v1.json'
            }
        }
    
    def _load_component(self, filename):
        """Load a component from file"""
        filepath = self.model_dir / filename
        if not filepath.exists():
            raise FileNotFoundError(f"File not found: {filepath}")
        
        try:
            return joblib.load(filepath)
        except Exception as e:
            raise RuntimeError(f"Failed to load {filename}: {e}")
    
    def preprocess(self, input_data, fill_missing=True):
        """
        Preprocess input data using JSON configuration
        
        Args:
            input_data: Dict, DataFrame, or list
            fill_missing: Whether to fill missing features with defaults
            
        Returns:
            Preprocessed numpy array
        """
        # Convert to DataFrame
        df = self._to_dataframe(input_data)
        
        if fill_missing:
            df = self._fill_missing_features(df)
        
        # Ensure correct column order
        df = df[self.feature_columns]
        
        return df.values
    
    def _to_dataframe(self, input_data):
        """Convert input to DataFrame"""
        if isinstance(input_data, dict):
            return pd.DataFrame([input_data])
        elif isinstance(input_data, list):
            return pd.DataFrame(input_data)
        elif isinstance(input_data, pd.DataFrame):
            return input_data.copy()
        else:
            raise ValueError("Input must be dict, list of dicts, or DataFrame")
    
    def _fill_missing_features(self, df):
        """Fill missing features with default values"""
        for feature in self.feature_columns:
            if feature not in df.columns:
                default_value = self.default_values.get(feature, 0.0)
                df[feature] = default_value
                print(f"  ⚠ Added missing feature: {feature} = {default_value}")
        return df
    
    def predict(self, input_data, return_details=True):
        """
        Make prediction with detailed output
        
        Args:
            input_data: Input data
            return_details: Return detailed information
            
        Returns:
            Prediction results
        """
        if not self.is_loaded:
            raise RuntimeError("Predictor not initialized")
        
        # Preprocess
        X = self.preprocess(input_data)
        
        # Scale
        X_scaled = self.scaler.transform(X)
        
        # Predict
        predictions = self.model.predict(X_scaled)
        
        # Prepare results
        results = []
        for i, pred in enumerate(predictions):
            result = {
                'sample_index': i,
                'prediction': int(pred),
                'label': 'FAILURE_WARNING' if pred == 1 else 'HEALTHY'
            }
            
            if return_details:
                if hasattr(self.model, 'predict_proba'):
                    proba = self.model.predict_proba(X_scaled)[i]
                    result['probabilities'] = {
                        'healthy': float(proba[0]),
                        'warning': float(proba[1])
                    }
                    result['confidence'] = float(max(proba))
                
                # Add feature summary for this sample
                if len(X) > 0:
                    result['feature_summary'] = {
                        'rms_value': float(X[i, 0]) if X.shape[1] > 0 else 0.0,
                        'peak_value': float(X[i, 3]) if X.shape[1] > 3 else 0.0,
                        'kurtosis': float(X[i, 1]) if X.shape[1] > 1 else 0.0
                    }
            
            results.append(result)
        
        return results if len(results) > 1 else results[0]
    
    def get_feature_info(self):
        """Get feature information"""
        return {
            'count': len(self.feature_columns),
            'columns': self.feature_columns,
            'categories': self.features_data.get('feature_categories', {}),
            'descriptions': self.feature_descriptions,
            'defaults': self.default_values
        }
    
    def validate_input(self, input_data):
        """Validate input features"""
        df = self._to_dataframe(input_data)
        
        missing = [f for f in self.feature_columns if f not in df.columns]
        extra = [col for col in df.columns if col not in self.feature_columns]
        
        return {
            'is_valid': len(missing) == 0,
            'missing_features': missing,
            'extra_features': extra,
            'provided_features': list(df.columns),
            'required_features': self.feature_columns,
            'match_percentage': (len(df.columns) - len(missing)) / len(self.feature_columns) * 100
        }
    
    def get_model_info(self):
        """Get complete model information"""
        return {
            'model': self.config['model'],
            'performance': self.config.get('performance', {}),
            'features': self.get_feature_info(),
            'files': self.config['files']
        }


# Django singleton instance
_predictor = None

def get_predictor(config_file="ml_config.json"):
    """Get singleton predictor instance for Django"""
    global _predictor
    if _predictor is None:
        try:
            _predictor = BearingFailurePredictor(config_file=config_file)
        except Exception as e:
            print(f"❌ Failed to initialize predictor: {e}")
            _predictor = None
    return _predictor


# Example Django view usage
"""
from django.http import JsonResponse
from .ml_predictor import get_predictor

def api_predict(request):
    predictor = get_predictor()
    if not predictor:
        return JsonResponse({'error': 'Model not loaded'}, status=500)
    
    data = request.data
    result = predictor.predict(data)
    return JsonResponse({'result': result})

def api_features(request):
    predictor = get_predictor()
    if not predictor:
        return JsonResponse({'error': 'Model not loaded'}, status=500)
    
    feature_info = predictor.get_feature_info()
    return JsonResponse({'features': feature_info})
"""
'''
    
    with open("ml_predictor.py", "w", encoding='utf-8') as f:
        f.write(script)

def create_django_config_file(feature_columns, version):
    """Create Django configuration file"""
    config = {
        'project': 'Bearing Failure Detection System',
        'version': version,
        'created': datetime.now().isoformat(),
        
        'model_config': {
            'name': 'BearingFailureDetector',
            'type': 'RandomForest',  # This will be updated based on actual model
            'input_features': len(feature_columns),
            'output_classes': 2,
            'thresholds': {
                'warning': 0.7,
                'critical': 0.9
            }
        },
        
        'file_structure': {
            'model': f'model_{version}.joblib',
            'scaler': f'scaler_{version}.joblib',
            'features_json': f'features_{version}.json',
            'features_simple': f'features_list_{version}.json',
            'metadata': f'model_metadata_{version}.json',
            'package': f'model_package_{version}.joblib'
        },
        
        'api_endpoints': {
            'predict': '/api/predict/',
            'features': '/api/features/',
            'health': '/api/health/',
            'model_info': '/api/model/info/'
        },
        
        'deployment': {
            'recommended_dir': 'ml_models/',
            'loading_class': 'BearingFailurePredictor',
            'requirements': 'requirements_django.txt'
        }
    }
    
    with open("ml_config.json", "w", encoding='utf-8') as f:
        json.dump(config, f, indent=4, ensure_ascii=False)
    
    print(f"✓ Django config saved: ml_config.json")

In [11]:
# ============================================================================
# EXECUTE THE SAVING PROCESS
# ============================================================================

print("\n" + "="*60)
print("STARTING MODEL SAVE WITH JSON FEATURES")
print("="*60)

# Ensure X_train and X_test are defined (from earlier training)
if 'X_train' not in locals():
    # If not defined, create dummy split (you should use your actual split)
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

# Save all components with JSON features
saved_files = save_model_components_with_json(
    results=results,
    scaler=scaler,
    feature_columns=feature_columns,
    df_metadata=df
)

print("\n" + "="*60)
print("FINAL FILE STRUCTURE")
print("="*60)
print("""
✅ BINARY FILES (for ML):
├── model_v1.joblib           # Trained ML model
├── scaler_v1.joblib          # Feature scaler
└── model_package_v1.joblib   # Complete package (backup)

✅ JSON FILES (for configuration):
├── features_v1.json          # Detailed feature info
├── features_list_v1.json     # Simple feature list
├── model_metadata_v1.json    # Complete metadata
└── ml_config.json           # Django configuration

✅ SUPPORT FILES:
├── requirements_django.txt   # Python requirements
└── ml_predictor.py          # Django loader class
""")


STARTING MODEL SAVE WITH JSON FEATURES

SAVING MODEL COMPONENTS (JSON FORMAT)
✓ Best model: GradientBoosting
  F1-Score: 0.8313
  Accuracy: 0.9854
✓ Scaler saved: scaler_v1.joblib
✓ Model saved: model_v1.joblib
✓ Features saved (JSON): features_v1.json
✓ Simple features list saved: features_list_v1.json


NameError: name 'sys' is not defined